<a href="https://colab.research.google.com/github/RolandoLopez16/RegresionLinealHV/blob/main/RegresionLinealHV_v8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresión Lineal HV v8 — EDA, preprocesamiento y simulador de escenarios

Este notebook aplica regresión lineal para analizar la productividad de equipos harvester usando únicamente las variables definidas en el documento del proyecto. La lógica es clara: primero se entiende el dato, luego se prepara de forma controlada, después se entrena el modelo y finalmente se usa como simulador de escenarios para comparar equipos bajo las mismas condiciones.


## 1. Librerías y configuración
Se cargan las librerías necesarias para análisis, gráficos, preprocesamiento, modelado y evaluación.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42


## 2. Carga del dataset
El archivo se carga desde local, Colab o GitHub RAW. Esto permite ejecutar el notebook en diferentes entornos sin cambiar el código.


In [ ]:
FILE_NAME = "Dataset_HV_2026_v2.xlsx"
GITHUB_URL = "https://raw.githubusercontent.com/RolandoLopez16/RegresionLinealHV/main/Dataset_HV_2026_v2.xlsx"

possible_paths = [
    f"/mnt/data/{FILE_NAME}",
    FILE_NAME,
    f"./{FILE_NAME}",
    f"/content/{FILE_NAME}",
]

DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)

if DATA_PATH is not None:
    print(f"Dataset cargado desde archivo local: {DATA_PATH}")
    df = pd.read_excel(DATA_PATH)
else:
    print("Dataset local no encontrado. Cargando desde GitHub RAW...")
    DATA_PATH = GITHUB_URL
    df = pd.read_excel(DATA_PATH)

print(f"Fuente utilizada: {DATA_PATH}")
print(f"Dimensiones iniciales: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()


## 3. Variables definidas por criterio de negocio
Se usan solo las variables señaladas en el documento para evitar que el modelo aprenda ruido o memorice ubicaciones específicas.


In [ ]:
variables_categoricas = [
    "EQUIPO",
    "MARCA",
    "ZONA",
    "CONTRATISTA",
    "ESPECIE",
    "TURNO",
    "SUELO",
]

variables_numericas = [
    "PENDIENTE PROMEDIO FINCA",
    "TOTAL DE ARBOLES",
    "DIAMETRO",
    "T PROGRAMADO",
    "HORAS DE OTRAS PARADA",
    "clima_temp_promedio_dia_c",
    "clima_temp_min_dia_c",
    "clima_temp_max_dia_c",
    "clima_precipitacion_dia_mm",
    "clima_viento_promedio_dia_kmh",
]

target = "M3/HORA"

columnas_requeridas = variables_categoricas + variables_numericas + [target]
columnas_faltantes = [c for c in columnas_requeridas if c not in df.columns]

if columnas_faltantes:
    raise ValueError(f"Faltan columnas requeridas en el dataset: {columnas_faltantes}")

print("Todas las columnas requeridas están disponibles.")


## 4. Construcción del dataset de modelado
Se crea una tabla de trabajo únicamente con las variables aprobadas para el análisis.


In [ ]:
df_model = df[columnas_requeridas].copy()

print(f"Dataset de modelado: {df_model.shape[0]} filas x {df_model.shape[1]} columnas")
df_model.head()


## 5. Diagnóstico inicial de calidad
Se revisan tipos de datos, valores faltantes y duplicados antes de cualquier transformación.


In [ ]:
resumen_calidad = pd.DataFrame({
    "tipo_dato": df_model.dtypes.astype(str),
    "nulos": df_model.isna().sum(),
    "%_nulos": (df_model.isna().mean() * 100).round(2),
    "unicos": df_model.nunique(dropna=True)
}).sort_values(by="%_nulos", ascending=False)

print("Duplicados exactos:", df_model.duplicated().sum())
resumen_calidad


## 6. Limpieza básica de texto y tipos
Se normalizan categorías y se convierten variables numéricas para evitar errores por formatos inconsistentes.


In [ ]:
for col in variables_categoricas:
    df_model[col] = (
        df_model[col]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace({"": np.nan, "NAN": np.nan, "NONE": np.nan, "NULL": np.nan})
    )

for col in variables_numericas + [target]:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

filas_antes = len(df_model)
df_model = df_model.dropna(subset=[target])
print("Filas eliminadas por target nulo:", filas_antes - len(df_model))

filas_antes = len(df_model)
df_model = df_model.drop_duplicates()
print("Duplicados eliminados:", filas_antes - len(df_model))

print("Dimensiones después de limpieza básica:", df_model.shape)


## 7. Validación operativa de la variable objetivo
La productividad debe ser positiva. Valores iguales o menores a cero no son útiles para este modelo.


In [ ]:
filas_antes = len(df_model)
df_model = df_model[df_model[target] > 0].copy()
print("Registros eliminados por productividad <= 0:", filas_antes - len(df_model))
print("Dimensiones:", df_model.shape)


## 8. EDA de la productividad
Se revisa la distribución de la variable objetivo para identificar dispersión, sesgo y posibles valores extremos.


In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df_model[target], bins=35)
plt.title("Distribución de la productividad M3/HORA")
plt.xlabel("M3/HORA")
plt.ylabel("Frecuencia")
plt.show()

plt.figure(figsize=(7,3))
plt.boxplot(df_model[target].dropna(), vert=False)
plt.title("Boxplot de productividad M3/HORA")
plt.xlabel("M3/HORA")
plt.show()

df_model[target].describe()


## 9. Análisis de outliers en productividad
Se identifican valores extremos mediante IQR. Para el modelo base se trabaja con el rango operacional típico, evitando que pocos extremos deformen la regresión lineal.


In [ ]:
Q1 = df_model[target].quantile(0.25)
Q3 = df_model[target].quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

outliers_target = df_model[(df_model[target] < limite_inf) | (df_model[target] > limite_sup)]
print(f"Límite inferior: {limite_inf:.3f}")
print(f"Límite superior: {limite_sup:.3f}")
print("Outliers detectados en productividad:", len(outliers_target))

df_model = df_model[(df_model[target] >= limite_inf) & (df_model[target] <= limite_sup)].copy()
print("Dimensiones después de controlar outliers del target:", df_model.shape)


## 10. EDA de variables numéricas
Se observa la distribución de cada variable numérica para comprender su escala y detectar comportamientos atípicos.


In [ ]:
for col in variables_numericas:
    plt.figure(figsize=(7,3))
    plt.hist(df_model[col].dropna(), bins=30)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()


## 11. Relación numérica con productividad
Los gráficos de dispersión permiten revisar si la relación con la productividad parece lineal, débil o afectada por ruido operativo.


In [ ]:
for col in variables_numericas:
    plt.figure(figsize=(7,4))
    plt.scatter(df_model[col], df_model[target], alpha=0.35)
    plt.title(f"{col} vs {target}")
    plt.xlabel(col)
    plt.ylabel(target)
    plt.show()


## 12. Correlación entre variables numéricas
La matriz de correlación ayuda a detectar variables relacionadas entre sí y posibles problemas de multicolinealidad.


In [ ]:
corr = df_model[variables_numericas + [target]].corr(numeric_only=True)

plt.figure(figsize=(10,8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlación")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

corr[target].sort_values(ascending=False)


## 13. EDA de variables categóricas
Se revisa la cardinalidad y la cantidad de registros por categoría. Esto es clave porque categorías con pocos datos pueden volver inestable la interpretación.


In [ ]:
for col in variables_categoricas:
    conteo = df_model[col].value_counts(dropna=False)
    print("\n", "="*80)
    print(col)
    print("Categorías:", df_model[col].nunique(dropna=True))
    display(conteo.head(15).to_frame("frecuencia"))

    plt.figure(figsize=(9,4))
    conteo.head(12).sort_values().plot(kind="barh")
    plt.title(f"Frecuencia por {col}")
    plt.xlabel("Registros")
    plt.ylabel(col)
    plt.show()


## 14. Productividad por variable categórica
Los boxplots muestran diferencias de productividad por equipo, marca, zona, contratista, especie, turno y suelo.


In [ ]:
for col in variables_categoricas:
    top_categorias = df_model[col].value_counts().head(12).index
    data_plot = df_model[df_model[col].isin(top_categorias)].copy()

    plt.figure(figsize=(11,5))
    data_plot.boxplot(column=target, by=col, rot=45)
    plt.title(f"Productividad por {col}")
    plt.suptitle("")
    plt.xlabel(col)
    plt.ylabel(target)
    plt.tight_layout()
    plt.show()


## 15. Preprocesamiento: agrupación de categorías poco frecuentes
Se agrupan categorías con baja frecuencia en `OTROS` para reducir ruido, controlar sobreajuste y mejorar la estabilidad del modelo.


In [ ]:
def agrupar_categorias_raras(data, columnas, min_registros=15):
    data = data.copy()
    mapa_categorias = {}

    for col in columnas:
        frecuencias = data[col].value_counts(dropna=True)
        categorias_validas = frecuencias[frecuencias >= min_registros].index
        data[col] = data[col].where(data[col].isin(categorias_validas), "OTROS")
        mapa_categorias[col] = list(categorias_validas)

    return data, mapa_categorias

df_model, mapa_categorias = agrupar_categorias_raras(
    df_model,
    variables_categoricas,
    min_registros=15
)

for col in variables_categoricas:
    print(col, "->", df_model[col].nunique(dropna=True), "categorías después de agrupar")


## 16. Preprocesamiento: control de outliers en variables numéricas
Se aplica winsorización a predictores numéricos para reducir el efecto de valores extremos sin eliminar registros.


In [ ]:
def winsorizar_columnas(data, columnas, p_inf=0.01, p_sup=0.99):
    data = data.copy()
    limites = {}

    for col in columnas:
        li = data[col].quantile(p_inf)
        ls = data[col].quantile(p_sup)
        data[col] = data[col].clip(lower=li, upper=ls)
        limites[col] = (li, ls)

    return data, limites

df_model, limites_winsor = winsorizar_columnas(df_model, variables_numericas)

pd.DataFrame(limites_winsor, index=["p01", "p99"]).T


## 17. División entrenamiento-prueba
Se separan los datos para evaluar el modelo en registros que no fueron usados durante el entrenamiento.


In [ ]:
X = df_model[variables_categoricas + variables_numericas].copy()
y = df_model[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


## 18. Pipeline de preprocesamiento
Las variables numéricas se imputan y estandarizan. Las categóricas se imputan y transforman con One-Hot Encoding.


In [ ]:
pipeline_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

pipeline_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, variables_numericas),
    ("cat", pipeline_categorico, variables_categoricas)
])

model = Pipeline(steps=[
    ("preprocess", preprocesador),
    ("model", LinearRegression())
])

model


## 19. Entrenamiento del modelo
Se ajusta una regresión lineal múltiple para explicar la productividad a partir de variables operativas y de contexto.


In [ ]:
model.fit(X_train, y_train)
print("Modelo entrenado correctamente.")


## 20. Evaluación en entrenamiento y prueba
Se calculan MAE, RMSE y R² para revisar error promedio, errores grandes y capacidad explicativa.


In [ ]:
def evaluar_modelo(modelo, X_data, y_real, nombre="datos"):
    y_pred = modelo.predict(X_data)
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    return pd.DataFrame({
        "conjunto": [nombre],
        "MAE": [mae],
        "RMSE": [rmse],
        "R2": [r2]
    })

metricas = pd.concat([
    evaluar_modelo(model, X_train, y_train, "Entrenamiento"),
    evaluar_modelo(model, X_test, y_test, "Prueba")
], ignore_index=True)

metricas


## 21. Validación cruzada
La validación cruzada revisa si el desempeño es estable y no depende de una sola partición entrenamiento-prueba.


In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

cv_results = cross_validate(
    model,
    X,
    y,
    cv=kf,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    return_train_score=False
)

resumen_cv = pd.DataFrame({
    "metrica": ["MAE", "RMSE", "R2"],
    "promedio": [
        -cv_results["test_MAE"].mean(),
        -cv_results["test_RMSE"].mean(),
        cv_results["test_R2"].mean()
    ],
    "desviacion": [
        cv_results["test_MAE"].std(),
        cv_results["test_RMSE"].std(),
        cv_results["test_R2"].std()
    ]
})

resumen_cv


## 22. Diagnóstico de residuos
Los residuos ayudan a revisar si el modelo comete errores sistemáticos o si los errores se distribuyen de forma razonable.


In [ ]:
y_pred_test = model.predict(X_test)
residuos = y_test - y_pred_test

plt.figure(figsize=(7,4))
plt.scatter(y_pred_test, residuos, alpha=0.45)
plt.axhline(0)
plt.title("Residuos vs predicciones")
plt.xlabel("Productividad estimada")
plt.ylabel("Residuo")
plt.show()

plt.figure(figsize=(7,4))
plt.hist(residuos, bins=30)
plt.title("Distribución de residuos")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.show()

pd.Series(residuos).describe()


## 23. Importancia de variables por permutación
Esta es la lectura principal para saber qué variables tienen más peso. Mide cuánto se deteriora el modelo cuando se altera una variable.


In [ ]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error"
)

importancia_variables = pd.DataFrame({
    "variable": X_test.columns,
    "importancia_MAE": perm.importances_mean,
    "desviacion": perm.importances_std
}).sort_values(by="importancia_MAE", ascending=False)

importancia_variables


## 24. Gráfico de importancia de variables
Mientras más alta la importancia, mayor peso tiene la variable en el desempeño predictivo del modelo.


In [ ]:
plot_imp = importancia_variables.sort_values("importancia_MAE", ascending=True)

plt.figure(figsize=(9,6))
plt.barh(plot_imp["variable"], plot_imp["importancia_MAE"])
plt.title("Importancia de variables por permutación")
plt.xlabel("Aumento del error MAE al alterar la variable")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()


## 25. Coeficientes del modelo
Los coeficientes muestran dirección del efecto. Se usan como apoyo interpretativo, no como única medida de importancia.


In [ ]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coeficientes = pd.DataFrame({
    "variable_transformada": feature_names,
    "coeficiente": model.named_steps["model"].coef_
})

coeficientes["variable_transformada"] = (
    coeficientes["variable_transformada"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

top_coeficientes = pd.concat([
    coeficientes.sort_values("coeficiente", ascending=False).head(10),
    coeficientes.sort_values("coeficiente", ascending=True).head(10)
])

top_coeficientes


## 26. Respuesta a la pregunta de negocio
La regresión lineal permite identificar qué variables operativas y de contexto influyen más en la productividad. La importancia por permutación indica peso predictivo; los coeficientes indican dirección e interpretación relativa.


In [ ]:
print("Variables con mayor peso según importancia por permutación:")
display(importancia_variables.head(10))

print("Coeficientes positivos y negativos más fuertes:")
display(top_coeficientes)


## 27. Simulador de escenarios
El simulador compara equipos manteniendo constantes las demás condiciones. Así se evita concluir que un equipo es mejor sin controlar zona, especie, suelo, turno y demás variables.


In [ ]:
def valor_tipico_escenario(data, variables_cat, variables_num):
    escenario = {}

    for col in variables_cat:
        moda = data[col].mode(dropna=True)
        escenario[col] = moda.iloc[0] if len(moda) > 0 else "OTROS"

    for col in variables_num:
        escenario[col] = data[col].median()

    return escenario

escenario_base = valor_tipico_escenario(df_model, variables_categoricas, variables_numericas)
escenario_base


## 28. Ajuste manual del escenario
Modifica estos valores para simular una operación específica. El modelo comparará los equipos bajo estas mismas condiciones.


In [ ]:
escenario_usuario = escenario_base.copy()

escenario_usuario.update({
    "MARCA": escenario_base["MARCA"],
    "ZONA": escenario_base["ZONA"],
    "CONTRATISTA": escenario_base["CONTRATISTA"],
    "ESPECIE": escenario_base["ESPECIE"],
    "TURNO": escenario_base["TURNO"],
    "SUELO": escenario_base["SUELO"],
    "PENDIENTE PROMEDIO FINCA": escenario_base["PENDIENTE PROMEDIO FINCA"],
    "TOTAL DE ARBOLES": escenario_base["TOTAL DE ARBOLES"],
    "DIAMETRO": escenario_base["DIAMETRO"],
    "T PROGRAMADO": escenario_base["T PROGRAMADO"],
    "HORAS DE OTRAS PARADA": escenario_base["HORAS DE OTRAS PARADA"],
    "clima_temp_promedio_dia_c": escenario_base["clima_temp_promedio_dia_c"],
    "clima_temp_min_dia_c": escenario_base["clima_temp_min_dia_c"],
    "clima_temp_max_dia_c": escenario_base["clima_temp_max_dia_c"],
    "clima_precipitacion_dia_mm": escenario_base["clima_precipitacion_dia_mm"],
    "clima_viento_promedio_dia_kmh": escenario_base["clima_viento_promedio_dia_kmh"],
})

escenario_usuario


## 29. Ranking de equipos según escenario
Se cambia únicamente el equipo y se mantiene constante el resto de variables. El mayor valor estimado representa el mejor equipo para ese escenario específico.


In [ ]:
def comparar_equipos(modelo, data, escenario, equipo_col="EQUIPO"):
    equipos = sorted(data[equipo_col].dropna().unique())
    resultados = []

    for equipo in equipos:
        fila = escenario.copy()
        fila[equipo_col] = equipo
        X_sim = pd.DataFrame([fila])[variables_categoricas + variables_numericas]
        pred = modelo.predict(X_sim)[0]
        resultados.append({
            "EQUIPO": equipo,
            "PRODUCTIVIDAD_ESTIMADA_M3_H": pred
        })

    return pd.DataFrame(resultados).sort_values(
        by="PRODUCTIVIDAD_ESTIMADA_M3_H",
        ascending=False
    ).reset_index(drop=True)

ranking_equipos = comparar_equipos(model, df_model, escenario_usuario)
ranking_equipos


## 30. Visualización del ranking de equipos
El gráfico permite comparar rápidamente los equipos para el escenario definido.


In [ ]:
top_ranking = ranking_equipos.head(15).sort_values("PRODUCTIVIDAD_ESTIMADA_M3_H", ascending=True)

plt.figure(figsize=(9,6))
plt.barh(top_ranking["EQUIPO"], top_ranking["PRODUCTIVIDAD_ESTIMADA_M3_H"])
plt.title("Ranking de equipos según escenario simulado")
plt.xlabel("Productividad estimada M3/HORA")
plt.ylabel("Equipo")
plt.tight_layout()
plt.show()


## 31. Interpretación del simulador
El simulador no dice que un equipo sea universalmente mejor. Indica cuál equipo tendría mayor productividad esperada bajo un conjunto específico de condiciones operativas.

La lectura correcta es:

- Si se mantienen constantes zona, contratista, especie, turno, suelo, pendiente, árboles, diámetro, tiempo programado, paradas y clima, el ranking compara el efecto esperado de cambiar únicamente el equipo.
- Si cambia el escenario, puede cambiar el mejor equipo.
- La decisión operativa debe combinar la predicción con experiencia técnica, disponibilidad de maquinaria y restricciones reales de campo.


## 32. Conclusión técnica
Esta versión v8 fortalece el análisis porque:

1. Usa variables definidas por criterio de negocio.
2. Realiza EDA antes de modelar.
3. Controla datos faltantes, categorías raras y valores extremos.
4. Evalúa el modelo con prueba y validación cruzada.
5. Usa importancia por permutación para determinar variables con más peso.
6. Incluye un simulador para comparar equipos bajo condiciones equivalentes.

Con esto, la regresión lineal pasa de ser un ejercicio predictivo a una herramienta de interpretación y apoyo a decisiones operativas.
